# Stage 08: Foundations of AI Agents and Agentic Workflows
## Reference Architecture: AI Agents vs Agentic AI Systems

This reference notebook implements and contrasts the architectural paradigms introduced in Lecture 55:
1. **Single AI Agent Architecture**: Single entity with focused scope, bounded autonomy, and deterministic rule dispatch.
2. **Agentic AI Architecture**: Multi-agent collaborative network orchestrating the 4 Cognitive Pillars:
   - **Perception**: Dynamic telemetry and multi-source environment observation.
   - **Reasoning**: Central LLM/heuristic reasoning brain assessing global objectives.
   - **Action**: Coordinated cross-agent task execution and actuator dispatch.
   - **Learning**: Adaptive reinforcement feedback loop updating policies across iterations.


In [1]:
from typing import List, Dict, Optional, Tuple, Any
from dataclasses import dataclass, field
from enum import Enum
import numpy as np

print("Dependencies loaded successfully.")


Dependencies loaded successfully.


### 1. Single AI Agent Architecture (Bounded Autonomy)

A single AI agent is designed to execute an individual, bounded task without general cross-domain collaboration. It operates within explicit operational boundaries and predefined rule sets.


In [2]:
class AgentExecutionStatus(str, Enum):
    SUCCESS = "SUCCESS"
    REJECTED = "REJECTED"
    OUT_OF_SCOPE = "OUT_OF_SCOPE"


@dataclass
class SingleAgentResponse:
    status: AgentExecutionStatus
    data: Dict[str, Any]
    message: str


class BankingCustomerServiceAgent:
    """
    Production implementation of an individual AI agent with a single focus.
    Handles banking inquiries within strict rule-governed boundaries.
    """

    def __init__(self, accounts_database: Dict[str, float]) -> None:
        self._db: Dict[str, float] = accounts_database
        self._allowed_intents = {"balance_inquiry", "transaction_summary"}

    def execute_task(self, intent: str, account_id: str) -> SingleAgentResponse:
        """Executes single task with deterministic intent validation."""
        if intent not in self._allowed_intents:
            return SingleAgentResponse(
                status=AgentExecutionStatus.OUT_OF_SCOPE,
                data={},
                message=f"Intent '{intent}' exceeds the bounded scope of this agent."
            )

        if account_id not in self._db:
            return SingleAgentResponse(
                status=AgentExecutionStatus.REJECTED,
                data={},
                message=f"Account '{account_id}' does not exist in the records."
            )

        if intent == "balance_inquiry":
            balance = self._db[account_id]
            return SingleAgentResponse(
                status=AgentExecutionStatus.SUCCESS,
                data={"account_id": account_id, "balance": balance},
                message=f"Current balance for {account_id} is ${balance:,.2f}."
            )

        return SingleAgentResponse(
            status=AgentExecutionStatus.SUCCESS,
            data={"account_id": account_id, "transactions_retrieved": 5},
            message=f"Retrieved 5 recent ledger entries for account {account_id}."
        )


# Verification of Single Agent Behavior
accounts_data = {"ACC_1001": 15420.50, "ACC_1002": 3200.00}
single_agent = BankingCustomerServiceAgent(accounts_database=accounts_data)

res_valid = single_agent.execute_task(intent="balance_inquiry", account_id="ACC_1001")
assert res_valid.status == AgentExecutionStatus.SUCCESS
assert res_valid.data["balance"] == 15420.50

res_invalid_intent = single_agent.execute_task(intent="apply_loan", account_id="ACC_1001")
assert res_invalid_intent.status == AgentExecutionStatus.OUT_OF_SCOPE

res_invalid_acc = single_agent.execute_task(intent="balance_inquiry", account_id="ACC_UNKNOWN")
assert res_invalid_acc.status == AgentExecutionStatus.REJECTED

print(f"Single Agent Test 1: {res_valid.message}")
print(f"Single Agent Test 2 (Out of Scope): {res_invalid_intent.message}")
print(f"Single Agent Test 3 (Rejected): {res_invalid_acc.message}")


Single Agent Test 1: Current balance for ACC_1001 is $15,420.50.
Single Agent Test 2 (Out of Scope): Intent 'apply_loan' exceeds the bounded scope of this agent.
Single Agent Test 3 (Rejected): Account 'ACC_UNKNOWN' does not exist in the records.


### 2. Agentic AI System Architecture (The 4 Cognitive Pillars)

Unlike a single-purpose agent, an Agentic AI system is a coordinated multi-agent network optimizing a global utility function. It cyclically processes:
1. **Perception**: Telemetry gathering from multiple sensory or environment sources.
2. **Reasoning**: Evaluating state transitions, constraint tradeoffs, and multi-agent coordination.
3. **Action**: Dispatching discrete, coordinated commands across all sub-agents.
4. **Learning**: Calculating policy error and adapting control parameters across operational cycles.


In [3]:
@dataclass
class EnvironmentalTelemetry:
    room_occupied: bool
    ambient_temp_c: float
    target_temp_c: float
    grid_peak_hours: bool
    current_power_consumption_kw: float


@dataclass
class AgenticDecisions:
    lighting_power_pct: float
    thermostat_mode: str
    target_temp_setting_c: float
    appliance_deferral: bool
    expected_power_kw: float


class LightingAgent:
    def evaluate(self, room_occupied: bool) -> float:
        return 100.0 if room_occupied else 0.0


class ThermostatAgent:
    def evaluate(self, room_occupied: bool, ambient_c: float, target_c: float, peak_hours: bool) -> Tuple[str, float]:
        if not room_occupied:
            return ("STANDBY", 26.0)
        if peak_hours:
            return ("ECO_COOL", target_c + 1.5)
        return ("COMFORT_COOL", target_c)


class ApplianceAgent:
    def evaluate(self, peak_hours: bool, consumption_kw: float) -> bool:
        return peak_hours and (consumption_kw > 3.0)


class SmartFacilityAgenticSystem:
    """
    Production implementation of an Agentic AI System coordinating multiple agents
    over the 4 Cognitive Pillars: Perception -> Reasoning -> Action -> Learning.
    """

    def __init__(self, target_reduction_pct: float = 0.25) -> None:
        self.lighting_agent = LightingAgent()
        self.thermostat_agent = ThermostatAgent()
        self.appliance_agent = ApplianceAgent()
        self.target_reduction_pct: float = target_reduction_pct
        self.policy_learning_rate: float = 0.05
        self.eco_offset: float = 1.0
        self.adaptation_history: List[Dict[str, float]] = []

    # Cognitive Pillar 1: Perception
    def perceive(self, telemetry: EnvironmentalTelemetry) -> EnvironmentalTelemetry:
        assert telemetry.current_power_consumption_kw > 0.0
        return telemetry

    # Cognitive Pillar 2: Reasoning
    def reason(self, telemetry: EnvironmentalTelemetry) -> AgenticDecisions:
        light_pct = self.lighting_agent.evaluate(telemetry.room_occupied)
        thermo_mode, thermo_target = self.thermostat_agent.evaluate(
            room_occupied=telemetry.room_occupied,
            ambient_c=telemetry.ambient_temp_c,
            target_c=telemetry.target_temp_c,
            peak_hours=telemetry.grid_peak_hours
        )
        # Apply learned adaptation parameter
        if telemetry.grid_peak_hours:
            thermo_target += self.eco_offset

        defer_appliances = self.appliance_agent.evaluate(
            telemetry.grid_peak_hours,
            telemetry.current_power_consumption_kw
        )

        expected_kw = telemetry.current_power_consumption_kw
        if light_pct == 0.0:
            expected_kw -= 0.6
        if thermo_mode == "STANDBY":
            expected_kw -= 1.8
        elif "ECO" in thermo_mode:
            expected_kw -= 0.8
        if defer_appliances:
            expected_kw -= 1.2

        expected_kw = max(0.5, expected_kw)

        return AgenticDecisions(
            lighting_power_pct=light_pct,
            thermostat_mode=thermo_mode,
            target_temp_setting_c=thermo_target,
            appliance_deferral=defer_appliances,
            expected_power_kw=expected_kw
        )

    # Cognitive Pillar 3: Action
    def act(self, decisions: AgenticDecisions) -> Dict[str, Any]:
        return {
            "dispatched_lighting": f"{decisions.lighting_power_pct:.1f}%",
            "dispatched_thermostat": f"{decisions.thermostat_mode} at {decisions.target_temp_setting_c:.1f}C",
            "dispatched_appliances": "DEFERRED" if decisions.appliance_deferral else "NORMAL",
            "projected_draw_kw": decisions.expected_power_kw
        }

    # Cognitive Pillar 4: Learning & Adaptation
    def learn(self, initial_kw: float, result_kw: float) -> float:
        reduction_achieved = (initial_kw - result_kw) / initial_kw
        error = self.target_reduction_pct - reduction_achieved

        # Update learned offset dynamically via negative feedback
        self.eco_offset += self.policy_learning_rate * error
        self.eco_offset = max(0.5, min(3.0, self.eco_offset))

        record = {
            "initial_kw": initial_kw,
            "result_kw": result_kw,
            "reduction_achieved": reduction_achieved,
            "updated_eco_offset": self.eco_offset
        }
        self.adaptation_history.append(record)
        return reduction_achieved

    def run_cycle(self, telemetry: EnvironmentalTelemetry) -> Dict[str, Any]:
        perceived = self.perceive(telemetry)
        decisions = self.reason(perceived)
        actions = self.act(decisions)
        achieved = self.learn(telemetry.current_power_consumption_kw, decisions.expected_power_kw)

        return {
            "actions": actions,
            "reduction_pct": round(achieved * 100, 2),
            "current_eco_offset": round(self.eco_offset, 3)
        }


# Execute Multi-Agent Cognitive Simulation across 2 operational states
agentic_sys = SmartFacilityAgenticSystem(target_reduction_pct=0.35)

# Scenario A: Occupied, Peak Hours, High Load
state_a = EnvironmentalTelemetry(
    room_occupied=True,
    ambient_temp_c=34.0,
    target_temp_c=22.0,
    grid_peak_hours=True,
    current_power_consumption_kw=5.8
)
out_a = agentic_sys.run_cycle(state_a)
assert out_a["reduction_pct"] > 0.0

# Scenario B: Unoccupied Room, Peak Hours
state_b = EnvironmentalTelemetry(
    room_occupied=False,
    ambient_temp_c=32.0,
    target_temp_c=22.0,
    grid_peak_hours=True,
    current_power_consumption_kw=4.5
)
out_b = agentic_sys.run_cycle(state_b)
assert out_b["actions"]["dispatched_lighting"] == "0.0%"
assert "STANDBY" in out_b["actions"]["dispatched_thermostat"]

print("Cycle A Execution Result:")
for k, v in out_a["actions"].items():
    print(f"  {k}: {v}")
print(f"  Reduction achieved: {out_a['reduction_pct']}%")

print("\nCycle B Execution Result (Unoccupied Room):")
for k, v in out_b["actions"].items():
    print(f"  {k}: {v}")
print(f"  Reduction achieved: {out_b['reduction_pct']}%")
print(f"  Adaptive Learning Parameter (eco_offset): {out_b['current_eco_offset']}")


Cycle A Execution Result:
  dispatched_lighting: 100.0%
  dispatched_thermostat: ECO_COOL at 24.5C
  dispatched_appliances: DEFERRED
  projected_draw_kw: 3.8
  Reduction achieved: 34.48%

Cycle B Execution Result (Unoccupied Room):
  dispatched_lighting: 0.0%
  dispatched_thermostat: STANDBY at 27.0C
  dispatched_appliances: DEFERRED
  projected_draw_kw: 0.8999999999999997
  Reduction achieved: 80.0%
  Adaptive Learning Parameter (eco_offset): 0.978


### 3. Quantitative Comparison and Architectural Invariants

| Attribute | Single AI Agent | Agentic AI System |
| :--- | :--- | :--- |
| **Entities Involved** | 1 (Monolithic Handler) | N (Collaborative Specialized Agents) |
| **Decision Logic** | Deterministic Intent Mapping / Rules | Multi-objective Telemetry Optimization |
| **Cognitive Breadth** | Reactive Task Execution | Perception -> Reasoning -> Action -> Learning Loop |
| **Failure Mode** | Strict rejection upon out-of-scope intent | Adaptive task re-allocation and policy degradation |


In [4]:
# Verification of Structural Invariants
assert len(agentic_sys.adaptation_history) == 2
assert agentic_sys.adaptation_history[1]["reduction_achieved"] > agentic_sys.adaptation_history[0]["reduction_achieved"]
assert out_b["actions"]["dispatched_appliances"] == "DEFERRED"

print("All architectural invariants and programmatic assertions verified successfully.")


All architectural invariants and programmatic assertions verified successfully.
